In [49]:
import matplotlib.pyplot as plt
import torch 
import torch.nn.functional as f
char_dim=10
bloc_size=3
nb_ite=100
hidden_layer=200
batch_size=32
lr=0.1

In [50]:
words=open("names.txt",'r').read().splitlines()
index_chars={i+1:chr(i+ord("a")) for i in range (26)}
index_chars[0]='.'
chars_index={i:s for s,i in index_chars.items()}

In [51]:
def build_data(words,bloc_size):
    X,Y=[],[]
    for w in words:
        sample=[0]*bloc_size
        for ch in w + ".":
            ix=chars_index[ch]
            X.append(sample)
            Y.append(ix)
    #        print(''.join(index_chars[i] for i in sample),"--->",index_chars[ix])
            sample=sample[1:]+[ix]
    X=torch.tensor(X)
    Y=torch.tensor(Y)
    return(X,Y)
import random 
random.seed(42)
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
Xtr,Ytr=build_data(words[:n1],bloc_size)
Xdev,Ydev=build_data(words[n1:n2],bloc_size)
Xte,Yte=build_data(words[n2:],bloc_size)

In [52]:
tuner1=(5/3)/((char_dim*bloc_size)**0.5)
tuner2=(5/3)/((hidden_layer)**0.5)
g=torch.Generator().manual_seed(2147483647)
C=torch.randn((27,char_dim))
w1=torch.randn((char_dim*bloc_size,hidden_layer),generator=g)*tuner1
b1=torch.randn(200,generator=g)*0.1
w2=torch.randn((hidden_layer,27),generator=g)*tuner2
b2=torch.randn(27,generator=g)*0.1
bngain=torch.ones((1,hidden_layer))
bnbias=torch.zeros((1,hidden_layer))
bnmean=torch.zeros((1,hidden_layer))
bnstd=torch.ones((1,hidden_layer))
parameters=[C,w1,b1,w2,b2,bngain,bnbias]
for p in parameters:
    p.requires_grad=True

In [53]:
lre=torch.linspace(-3,0,1000)
lrs=10**lre
lossi,stepi=[],[]
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [54]:
emb = C[Xb] 
embcat = emb.view(emb.shape[0], -1)
hprebn = embcat @ w1 + b1 
bnmeani = 1/batch_size*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(batch_size-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ w2 + b2 
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(batch_size), Yb].mean()
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()

In [55]:
#boring backprop method
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(batch_size), Yb]=-1
dprobs=dlogprobs*probs**(-1)
dcounts=dprobs*counts_sum_inv
dcounts_sum_inv=(dprobs*counts).sum(1,keepdim=True)
dcounts_sum=-dcounts_sum_inv*(counts_sum_inv**2)
dcounts+=torch.ones_like(counts) * dcounts_sum
dnorm_logits=dcounts*counts
dlogits=dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += f.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh=dlogits@w2.T
dw2=h.T@dlogits
db2=dlogits.sum(0)
dhpreact=dh*(1-h**2)
dbngain=(dhpreact*bnraw).sum(0,keepdim=True)
dbnraw=dhpreact*bngain
dbnbias=dhpreact.sum(0,keepdim=True)
dbndiff=dbnraw*bnvar_inv
dbnvar_inv=(dbnraw*bndiff).sum(0,keepdim=True)
dbnvar=dbnvar_inv*(-0.5*bnvar_inv**3)
dbndiff2= 1/(batch_size-1)*torch.ones_like(bndiff2) * dbnvar
dbndiff+=2*dbndiff2*bndiff
dhprebn=dbndiff.clone()
dbmeani=-dbndiff.clone().sum(0)
dhprebn+=1/batch_size*torch.ones_like(hprebn) * dbmeani
dembcat=dhprebn@w1.T
dw1=embcat.T@dhprebn
db1=dhprebn.sum(0)
demb=dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]

In [56]:
#gigachad backprop method
dlogits=probs.clone()
dlogits[range(batch_size), Yb]+=-1
dhprebn1=(bngain*bnvar_inv/batch_size)*(batch_size*dhpreact-dhpreact.sum(0,keepdim=True)-(batch_size/(batch_size-1))*bnraw*(dhpreact*bnraw).sum(0,keepdim=True))
(dhprebn1-dhprebn).max()

tensor(2.9802e-08, grad_fn=<MaxBackward1>)